<div style="background-color: #222; padding: 24px;">
    <h1 style="color: #d4bbff; margin-bottom: 8px;">Medallion: Food Item Categorization</h1>
    <h3 style="color: #fff; margin-top: 0;">Categorize and export food items from bronze to gold layer.</h3>
</div>

In [11]:
#!/usr/bin/env python3
"""
Refactored: Bronze -> Silver -> Gold Food Item Categorization Pipeline using Delta Lake.
Reads bronze (json | delta | jdbc), cleans & categorizes, writes Silver (cleaned) and Gold (aggregates)
as Delta tables. Configurable via environment (.env).
"""
import os
import re
import logging
import shutil
import unicodedata
from dataclasses import dataclass
from typing import List, Optional

from dotenv import load_dotenv
from pyspark import SparkConf
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, DoubleType, TimestampType
)

# ---- Load env ----
BASE_DIR = os.getcwd()  # Jupyter's current notebook directory
load_dotenv(os.path.join(BASE_DIR, ".env"))

# ---- Logging ----
LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO").upper()
logging.basicConfig(level=LOG_LEVEL, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("food-pipeline")

# ---- Config dataclass ----
@dataclass
class PipelineConfig:
    bronze_path: str = os.getenv("BRONZE_PATH", "/home/jovyan/work/data/bronze")
    silver_path: str = os.getenv("SILVER_PATH", "/home/jovyan/work/data/silver")
    gold_path: str = os.getenv("GOLD_PATH", "/home/jovyan/work/data/gold")
    temp_output_path: str = os.getenv("TEMP_OUTPUT_PATH", "/home/jovyan/work/tmp")
    spark_master: str = os.getenv("SPARK_MASTER", "spark://spark-master:7077")
    app_name: str = os.getenv("SPARK_APP_NAME", "MedallionFoodPipeline")
    executor_memory: str = os.getenv("SPARK_EXECUTOR_MEMORY", "2g")
    driver_memory: str = os.getenv("SPARK_DRIVER_MEMORY", "2g")
    warehouse_dir: str = os.getenv("SPARK_WAREHOUSE_DIR", "/tmp/spark-warehouse")
    bronze_format: str = os.getenv("BRONZE_FORMAT", "json")  # options: json, delta, jdbc
    jdbc_table: Optional[str] = os.getenv("BRONZE_JDBC_TABLE")
    jdbc_url: Optional[str] = os.getenv("BRONZE_JDBC_URL")  # e.g. jdbc:postgresql://source-postgres:5432/postgres
    jdbc_user: Optional[str] = os.getenv("BRONZE_JDBC_USER", "postgres")
    jdbc_password: Optional[str] = os.getenv("BRONZE_JDBC_PASSWORD", "postgres")
    export_to_postgres: bool = os.getenv("EXPORT_TO_POSTGRES", "false").lower() in ("1", "true", "yes")
    export_postgres_table: str = os.getenv("EXPORT_POSTGRES_TABLE", "gold_food_aggregates")
    postgres_jdbc_url: Optional[str] = os.getenv("POSTGRES_JDBC_URL")
    postgres_user: Optional[str] = os.getenv("POSTGRES_USER")
    postgres_password: Optional[str] = os.getenv("POSTGRES_PASSWORD")
    coalesce_files: bool = os.getenv("COALESCE_FILES", "true").lower() in ("1", "true", "yes")
    partition_by_food_type: bool = os.getenv("PARTITION_BY_FOOD_TYPE", "true").lower() in ("1", "true", "yes")


# ---- Utilities ----
def verify_directory(path: str) -> None:
    os.makedirs(path, exist_ok=True)
    try:
        os.chmod(path, 0o777)
    except PermissionError:
        pass
    if not (os.access(path, os.R_OK) and os.access(path, os.W_OK)):
        raise PermissionError(f"Insufficient permissions for path: {path}")


def safe_clean_dir(path: str) -> None:
    if os.path.exists(path):
        for name in os.listdir(path):
            full = os.path.join(path, name)
            try:
                if os.path.isdir(full):
                    shutil.rmtree(full, ignore_errors=True)
                else:
                    os.remove(full)
            except Exception:
                pass
    else:
        os.makedirs(path, exist_ok=True)


# ---- Spark creation (Delta enabled) ----
def create_spark_session(cfg: PipelineConfig) -> SparkSession:
    conf = (
        SparkConf()
        .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .set("spark.hadoop.fs.permissions.umask-mode", "000")
        .set("spark.sql.sources.ignoreNonExistentPaths", "true")
        .set("spark.executor.extraJavaOptions", "-Djava.io.tmpdir=/tmp")
        .set("spark.driver.extraJavaOptions", "-Djava.io.tmpdir=/tmp")
        .set("spark.executor.memory", cfg.executor_memory)
        .set("spark.driver.memory", cfg.driver_memory)
        .set("spark.sql.warehouse.dir", cfg.warehouse_dir)
    )
    spark = (
        SparkSession.builder
        .config(conf=conf)
        .master(cfg.spark_master)
        .appName(cfg.app_name)
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel(LOG_LEVEL)
    return spark


# ---- Schemas ----
purchase_item_schema = StructType([
    StructField("Código", StringType(), True),
    StructField("Descrição", StringType(), True),
    StructField("Qtde", StringType(), True),
    StructField("Un", StringType(), True),
    StructField("Vl Unit", StringType(), True),
    StructField("Vl Total", StringType(), True),
])
json_schema = StructType([
    StructField("store", StringType(), True),
    StructField("cnpj", StringType(), True),
    StructField("store_state_code", StringType(), True),
    StructField("store_address", StringType(), True),
    StructField("purchase_date", StringType(), True),
    StructField("access_key", StringType(), True),
    StructField("purchase", ArrayType(purchase_item_schema), True),
])


# ---- Category lists (can be expanded or loaded from external resource) ----
VEGETABLES = ['berinjela', 'cebola', 'cenoura', 'tomate', 'abobora', 'pepino', 'rucula', 'batata', 'alface', 'brocolis', 'repolho', 'beterraba', 'mandioquinha']
MEATS = ['bife', 'file', 'coxinha', 'chuleta', 'bacon', 'patinho', 'peito', 'frango', 'carne', 'coxa', 'linguiça', 'guisado', 'atum']
DAIRY = ['leite', 'iogurte', 'queijo', 'mussarela', 'nata', 'margarina', 'manteiga', 'q.lanche']
BEVERAGES = ['coca-cola', 'agua', 'suco', 'cha', 'energi', 'monster', 'cafe', 'nescafe', 'vinho', 'cerveja', 'guarana','dolce gusto']
SEASONINGS = ['tempero', 'molho', 'sazon', 'mostarda', 'catchup', 'sal', 'oregano', 'paprica', 'chimichurri', 'cominho', 'coentro', 'maionese', 'ext.elefante', 'vinagre']
GRAINS = ['arroz', 'feijao', 'farinha', 'massa', 'pao', 'bolo', 'tapioca', 'milho', 'lentilha', 'aveia', 'grao', 'far.maria', 'feij.pto']
SNACKS = ['biscoito', 'chocolate', 'snickers', 'bombom', 'paodequeijo', 'gelatina', 'dulce', 'doce', 'barra', 'cookie', 'pipoca']
FRUITS = ['banana', 'laranja', 'maca', 'abacaxi', 'pera', 'uva', 'mamão', 'goiaba', 'manga', 'kiwi', 'ameixa', 'bergamota', 'tangerina', 'caqui', 'caju', 'morango']
NON_FOOD = ['amac.downy', 'esponja', 'papel hig', 'det.liq', 'sab liq', 'limpador', 'colgate', 'vela', 'toalha', 'algodao', 'abs', 'antartica', 'isquiero', 'suporte', 'coador', 'pinça', 'lixa', 'alicate', 'cortador', 'perfume', 'desodorante', 'sabonete', 'pente', 'escova', 'gel', 'repelente', 'pasta', 'creme', 'shampoo', 'condicionador']


def normalize_text(text: Optional[str]) -> str:
    if text is None:
        return ""
    txt = unicodedata.normalize("NFKD", str(text))
    txt = txt.encode("ASCII", "ignore").decode("utf-8")
    txt = txt.lower()
    txt = re.sub(r"[^a-z0-9\s\-\.]", " ", txt)
    return txt


def categorize_text(desc: Optional[str]) -> str:
    d = normalize_text(desc)
    if not d:
        return "Non-food"
    for k in NON_FOOD:
        if k in d:
            return "Non-food"
    for k in VEGETABLES:
        if k in d:
            return "Vegetable"
    for k in MEATS:
        if k in d:
            return "Meat"
    for k in DAIRY:
        if k in d:
            return "Dairy"
    for k in BEVERAGES:
        if k in d:
            return "Beverage"
    for k in SEASONINGS:
        if k in d:
            return "Seasoning"
    for k in GRAINS:
        if k in d:
            return "Grain"
    for k in SNACKS:
        if k in d:
            return "Snack"
    for k in FRUITS:
        if k in d:
            return "Fruit"
    return "Non-food"


CATEGORY_UDF = F.udf(categorize_text, StringType())


# ---- Pipeline class ----
class FoodPipeline:
    def __init__(self, spark: SparkSession, cfg: PipelineConfig):
        self.spark = spark
        self.cfg = cfg

    def read_bronze(self) -> DataFrame:
        fmt = self.cfg.bronze_format.lower()
        if fmt == "delta":
            log.info("Reading bronze as Delta: %s", self.cfg.bronze_path)
            return self.spark.read.format("delta").load(self.cfg.bronze_path)
        if fmt == "jdbc":
            if not (self.cfg.jdbc_url and self.cfg.jdbc_table):
                raise ValueError("jdbc_url and jdbc_table must be set for jdbc bronze_format")
            log.info("Reading bronze via JDBC: %s table %s", self.cfg.jdbc_url, self.cfg.jdbc_table)
            props = {"user": self.cfg.jdbc_user, "password": self.cfg.jdbc_password, "driver": "org.postgresql.Driver"}
            return self.spark.read.jdbc(url=self.cfg.jdbc_url, table=self.cfg.jdbc_table, properties=props)
        # default: json
        log.info("Reading bronze as JSON from: %s", self.cfg.bronze_path)
        return self.spark.read.schema(json_schema).option("multiLine", True).json(self.cfg.bronze_path)

    def clean_and_explode(self, df: DataFrame) -> DataFrame:
        df_exp = (
            df.select(
                F.col("store").alias("STORE"),
                F.to_timestamp(F.col("purchase_date"), "dd/MM/yyyy HH:mm:ss").alias("PURCHASED_AT"),
                F.explode(F.col("purchase")).alias("purchase_item")
            )
        )

        def col_from_item(field_name: str):
            return F.col("purchase_item").getItem(field_name)

        df_items = (
            df_exp.select(
                col_from_item("Código").alias("CD_ITEM"),
                col_from_item("Descrição").alias("DESCRIPTION"),
                col_from_item("Qtde").alias("QTDE_RAW"),
                col_from_item("Un").alias("UNIT"),
                col_from_item("Vl Unit").alias("VL_UNIT_RAW"),
                col_from_item("Vl Total").alias("VL_TOTAL_RAW"),
                F.col("STORE"),
                F.col("PURCHASED_AT"),
            )
        )

        # Clean numeric columns: replace comma with dot and remove thousands dot
        def clean_decimal_expr(col_expr):
            return F.regexp_replace(F.regexp_replace(col_expr, r"\.", ""), ",", ".").cast(DoubleType())

        df_cleaned = (
            df_items
            .withColumn("QTDE", clean_decimal_expr(F.col("QTDE_RAW")))
            .withColumn("VL_UNIT", clean_decimal_expr(F.col("VL_UNIT_RAW")))
            .withColumn("VL_TOTAL", clean_decimal_expr(F.col("VL_TOTAL_RAW")))
            .drop("QTDE_RAW", "VL_UNIT_RAW", "VL_TOTAL_RAW")
        )

        return df_cleaned

    def categorize(self, df: DataFrame) -> DataFrame:
        df2 = df.withColumn("FOOD_TYPE", CATEGORY_UDF(F.col("DESCRIPTION")))
        df2 = df2.withColumn("IS_FOOD", F.when(F.col("FOOD_TYPE") == "Non-food", F.lit("No")).otherwise(F.lit("Yes")))
        return df2

    def write_delta(self, df: DataFrame, path: str, partition_by: Optional[List[str]] = None) -> None:
        options = {"overwriteSchema": "true"}
        writer = df.write.format("delta").mode("overwrite").options(**options)
        if partition_by:
            writer = writer.partitionBy(*partition_by)
        if self.cfg.coalesce_files:
            # try to reduce small files in local dev
            try:
                writer = writer.coalesce(1)
            except Exception:
                pass
        writer.save(path)
        log.info("Wrote Delta to: %s", path)

    def build_silver(self, df_cleaned: DataFrame) -> DataFrame:
        df_silver = (
            df_cleaned
            .select(
                "CD_ITEM",
                "DESCRIPTION",
                "QTDE",
                "UNIT",
                "VL_UNIT",
                "VL_TOTAL",
                "STORE",
                "PURCHASED_AT"
            )
        )
        return df_silver

    def build_gold(self, df_silver: DataFrame) -> DataFrame:
        df_gold = (
            df_silver
            .groupBy("STORE", "FOOD_TYPE")
            .agg(
                F.count(F.lit(1)).alias("ROWS"),
                F.sum(F.coalesce(F.col("QTDE"), F.lit(0.0))).alias("TOTAL_QTDE"),
                F.sum(F.coalesce(F.col("VL_TOTAL"), F.lit(0.0))).alias("TOTAL_VALUE")
            )
        )
        return df_gold

    def export_to_postgres(self, df: DataFrame) -> None:
        if not (self.cfg.postgres_jdbc_url and self.cfg.postgres_user and self.cfg.postgres_password):
            log.warning("Postgres export is enabled but Postgres credentials/url are missing.")
            return
        props = {"user": self.cfg.postgres_user, "password": self.cfg.postgres_password, "driver": "org.postgresql.Driver"}
        df.write.jdbc(url=self.cfg.postgres_jdbc_url, table=self.cfg.export_postgres_table, mode="overwrite", properties=props)
        log.info("Exported Gold to Postgres table: %s", self.cfg.export_postgres_table)

    def run(self) -> None:
        log.info("Pipeline start. Bronze format=%s", self.cfg.bronze_format)
        df_raw = self.read_bronze()
        log.info("Raw bronze count estimate: %s", df_raw.count() if df_raw.rdd.getNumPartitions() > 0 else "unknown")

        df_cleaned = self.clean_and_explode(df_raw)
        df_categorized = self.categorize(df_cleaned)

        # Write Silver
        silver_path = os.path.join(self.cfg.silver_path, "food_items")
        verify_directory(os.path.dirname(silver_path))
        self.write_delta(df_categorized, silver_path, partition_by=["FOOD_TYPE"] if self.cfg.partition_by_food_type else None)

        # Build and write Gold aggregates
        df_gold = self.build_gold(df_categorized)
        gold_path = os.path.join(self.cfg.gold_path, "food_aggregates")
        verify_directory(os.path.dirname(gold_path))
        self.write_delta(df_gold, gold_path, partition_by=["STORE"] if self.cfg.partition_by_food_type else None)

        # Optional CSV snapshot for quick inspection (temp)
        tmp_csv = os.path.join(self.cfg.temp_output_path, "gold_snapshot")
        try:
            verify_directory(self.cfg.temp_output_path)
            safe_clean_dir(tmp_csv)
            if self.cfg.coalesce_files:
                df_gold.coalesce(1).write.mode("overwrite").option("header", True).csv(tmp_csv)
            else:
                df_gold.write.mode("overwrite").option("header", True).csv(tmp_csv)
            log.info("Gold CSV snapshot written to: %s", tmp_csv)
        except Exception:
            log.exception("Failed to write CSV snapshot; continuing.")

        log.info("Pipeline finished successfully.")


# ---- main ----
def main():
    cfg = PipelineConfig()
    for p in (cfg.bronze_path, cfg.silver_path, cfg.gold_path, cfg.temp_output_path):
        try:
            verify_directory(p)
        except Exception as e:
            log.error("Directory verification failed for %s: %s", p, e)
            raise

    spark = create_spark_session(cfg)
    pipeline = FoodPipeline(spark, cfg)
    pipeline.run()
    spark.stop()


if __name__ == "__main__":
    main()



2025-09-04 03:05:06,552 INFO Pipeline start. Bronze format=json
2025-09-04 03:05:06,552 INFO Reading bronze as JSON from: /home/jovyan/work/data/bronze


Py4JJavaError: An error occurred while calling o170.javaToPython.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'spark_catalog': org.apache.spark.sql.delta.catalog.DeltaCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1925)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.loadV2SessionCatalog(CatalogManager.scala:67)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$2(CatalogManager.scala:86)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$1(CatalogManager.scala:86)
	at scala.Option.map(Option.scala:230)
	at org.apache.spark.sql.connector.catalog.CatalogManager.v2SessionCatalog(CatalogManager.scala:85)
	at org.apache.spark.sql.connector.catalog.Cat